# PMI-based word vectors from Simpsons dialogue

This notebook reproduces `pmi-convo-vectorization-simpsons.py` (week3) but reads the raw Simpsons data files directly instead of using `convokit`.

It expects the following files to already be available locally in the Colab runtime, under `/content/sample_data/` (the standard Colab `sample_data` folder, visible in the Files pane):
- `simpsons_speakers.txt` (columns: `speaker_id`, `gender`)
- `simpsons_lines.txt` (columns: `line_id`, `speaker_id`, `text`)

Drag these two files directly into the `sample_data` folder in the Colab Files pane before running the cells below.

We then:
1. Split utterances into texts spoken by male vs. female characters.
2. Run a quick bigram collocation analysis (PMI) on the male utterances.
3. Build a word co-occurrence (skip-gram) matrix on the female utterances and convert it to a Positive Pointwise Mutual Information (PPMI) matrix.
4. Use the PPMI matrix as word vectors and look up the most similar words for a given query word.

In [ ]:
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
import numpy as np
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

nltk.download("stopwords")

# Reuses some ideas from https://www.kaggle.com/code/gabrielaltay/word-vectors-from-pmi-matrix/notebook

## 1. Load the data

We read the speakers file to build a `speaker_id -> gender` lookup, then read the lines file and split the utterance texts into `male` and `female` lists based on the speaker's gender. This replaces the `convokit.Corpus` used in the original script.

In [ ]:
SPEAKERS_PATH = "/content/sample_data/simpsons_speakers.txt"
LINES_PATH = "/content/sample_data/simpsons_lines.txt"


def load_speaker_genders(path):
    genders = {}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 2:
                continue
            speaker_id, gender = fields[0], fields[1]
            genders[speaker_id] = gender
    return genders


def process_data():
    print("Loading data...")
    genders = load_speaker_genders(SPEAKERS_PATH)

    male = []
    female = []
    with open(LINES_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            _line_id, speaker_id, text = fields[0], fields[1], fields[2]
            gender = genders.get(speaker_id, "?")
            if gender == "f":
                female.append(text)
            elif gender == "m":
                male.append(text)
    print("Data loaded.")
    print("No. male:", len(male), "No. female:", len(female))
    return male[:1000], female[:1000]


male, female = process_data()

## 2. Bigram collocation analysis (male utterances)

As a quick sanity check, we find the top 100 bigrams (by PMI) among the male utterances, keeping only bigrams that occur at least 3 times.

In [ ]:
bigram_measures = BigramAssocMeasures()

all_words = []
for text in male:
    all_words.extend(nltk.wordpunct_tokenize(text))

finder = BigramCollocationFinder.from_words(all_words)
finder.apply_freq_filter(3)  # must occur minimally 3 times

bigrams = finder.nbest(bigram_measures.pmi, 100)

with open("simpsons-bigrams-PMI.txt", "w") as bi_out:
    for bigram in bigrams:
        bi_out.write(str(bigram) + "\n")

print("Top 20 bigrams by PMI:")
for bigram in bigrams[:20]:
    print(bigram)
print("\nSee simpsons-bigrams-PMI.txt for the full top-100 list.")

## 3. Skip-gram co-occurrence counts and PPMI

These helper functions build:
- `priors`: raw word frequencies.
- `joints`: skip-gram (word, context-word) co-occurrence counts within a symmetric window.
- a Positive Pointwise Mutual Information (PPMI) matrix from those counts, which we use as word vectors.

This is the same logic as in the original script, just without the `convokit` dependency.

In [ ]:
def get_priors(texts):
    priors = Counter()
    for text in texts:
        for word in text.split(" "):
            priors[word] += 1
    return priors


def get_joints(texts, left, right, tok2indx):
    joints = Counter()
    for text in texts:
        tokens = [tok2indx[tok] for tok in text.split(" ")]
        for ii_word, _word in enumerate(tokens):
            ii_context_min = max(0, ii_word - left)
            ii_context_max = min(len(text) - 1, ii_word + right)
            ii_contexts = [ii for ii in range(ii_context_min, ii_context_max + 1) if ii != ii_word]
            for ii_context in ii_contexts:
                if ii_context > len(tokens) - 1:
                    break
                skipgram = (tokens[ii_word], tokens[ii_context])
                joints[skipgram] += 1
    return joints


def create_indexes(priors):
    tok2indx = {tok: indx for indx, tok in enumerate(priors.keys())}
    indx2tok = {indx: tok for tok, indx in tok2indx.items()}
    return tok2indx, indx2tok


def create_ppmi(priors, joints):
    row_indxs = []
    col_indxs = []
    dat_values = []
    for (tok1, tok2), sg_count in joints.items():
        row_indxs.append(tok1)
        col_indxs.append(tok2)
        dat_values.append(sg_count)
    wwcnt_mat = sparse.csr_matrix((dat_values, (row_indxs, col_indxs)))
    num_skipgrams = wwcnt_mat.sum()
    assert sum(joints.values()) == num_skipgrams

    row_indxs = []
    col_indxs = []
    ppmi_dat_values = []  # positive pointwise mutual information
    sum_over_words = np.array(wwcnt_mat.sum(axis=0)).flatten()
    sum_over_contexts = np.array(wwcnt_mat.sum(axis=1)).flatten()
    alpha = 0.75
    sum_over_words_alpha = sum_over_words ** alpha
    nca_denom = np.sum(sum_over_words_alpha)

    ppmi_mat = None
    for (tok_word, tok_context), sg_count in tqdm(joints.items()):
        nwc = sg_count
        Pwc = nwc / num_skipgrams
        nw = sum_over_contexts[tok_word]
        Pw = nw / num_skipgrams
        nc = sum_over_words[tok_context]
        Pc = nc / num_skipgrams
        nca = sum_over_words_alpha[tok_context]
        Pca = nca / nca_denom
        pmi = np.log2(Pwc / (Pw * Pc))
        ppmi = pmi  # max(pmi, 0)
        row_indxs.append(tok_word)
        col_indxs.append(tok_context)
        ppmi_dat_values.append(ppmi)
        ppmi_mat = sparse.csr_matrix((ppmi_dat_values, (row_indxs, col_indxs)))
    return ppmi_mat


def most_similar(word, mat, topn, tok2indx, indx2tok):
    indx = tok2indx[word]
    if isinstance(mat, sparse.csr_matrix):
        v1 = mat.getrow(indx)
    else:
        v1 = mat[indx:indx + 1, :]
    sims = cosine_similarity(mat, v1).flatten()
    sindxs = np.argsort(-sims)
    sim_word_scores = [(indx2tok[sindx], sims[sindx]) for sindx in sindxs[0:topn]]
    return sim_word_scores

## 4. Build the PPMI matrix (female utterances)

As in the original script, we build word vectors from the female utterances using a symmetric context window of 2 words on each side.

In [ ]:
with open("simpsons-texts.txt", "w") as out:
    for t in male:
        out.write("Male:" + t + "\n")
    for t in female:
        out.write("Female:" + t + "\n")
print("See simpsons-texts.txt")

priors = get_priors(female)  # male
print("Priors done.")
tok2indx, indx2tok = create_indexes(priors)
print("Indexes done.")
joints = get_joints(female, 2, 2, tok2indx)  # male
print("Joints done. Now computing PMI.")
mat = create_ppmi(priors, joints)
print("PPMI done.")

common = priors.most_common(10)
print("Most common words:", common)

## 5. Explore the word vectors

For each of the most frequent words, show the most similar words according to the PPMI vectors.

In [ ]:
for word, freq in common:
    similar = [w for w in most_similar(word, mat, 5, tok2indx, indx2tok) if w[0] != word]
    print(word, similar)

## 6. Try your own words

Run the cell below and type a word to see its most similar words (type `#` to stop).

In [ ]:
inp = ""
while inp != "#":
    inp = input("Type a word (# to stop): ")
    if inp == "#":
        break
    if inp not in tok2indx:
        print("Word not in lexicon.")
        continue
    print(inp, [w for w in most_similar(inp, mat, 25, tok2indx, indx2tok) if w[0] != inp])